In [1]:
# ============================================================
# CELL 1 — CONNECT TO DATABASE
# ============================================================

import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import warnings
warnings.filterwarnings('ignore')

DB_USER     = 'postgres'
DB_PASSWORD = quote_plus('NewStrongPassword@123')  # your actual password
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'franchise_intelligence'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

def run_query(sql):
    """Helper function — run any SQL and return a DataFrame"""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

# Test
result = run_query("SELECT COUNT(*) as total_transactions FROM daily_transactions")
print("✅ Connected to franchise_intelligence database")
print(f"Total transactions in DB: {result['total_transactions'].values[0]:,}")

✅ Connected to franchise_intelligence database
Total transactions in DB: 681,949


In [2]:
# ============================================================
# QUERY 1 — FRANCHISE REVENUE LEADERBOARD
# Business Question: Which outlets are our top and bottom
# revenue generators over the full 18 month period?
# ============================================================

q1 = """
SELECT 
    om.outlet_id,
    om.city,
    om.archetype,
    om.outlet_type,
    om.zone,
    ROUND(SUM(mp.revenue)::NUMERIC, 2)                    AS total_revenue,
    ROUND(AVG(mp.revenue)::NUMERIC, 2)                    AS avg_monthly_revenue,
    ROUND(SUM(mp.total_orders)::NUMERIC, 0)               AS total_orders,
    ROUND(AVG(mp.avg_order_value)::NUMERIC, 2)            AS avg_order_value,
    ROUND(AVG(mp.net_margin_pct)::NUMERIC, 2)             AS avg_net_margin_pct,
    ROUND(AVG(mp.avg_rating)::NUMERIC, 2)                 AS avg_rating,
    RANK() OVER (ORDER BY SUM(mp.revenue) DESC)           AS revenue_rank
FROM monthly_performance mp
JOIN outlet_master om ON mp.outlet_id = om.outlet_id
GROUP BY om.outlet_id, om.city, om.archetype, om.outlet_type, om.zone
ORDER BY total_revenue DESC
"""

q1_result = run_query(q1)

print("📊 QUERY 1 — FRANCHISE REVENUE LEADERBOARD")
print("="*65)
print(f"\nTOP 10 OUTLETS BY REVENUE:")
print(q1_result.head(10).to_string(index=False))
print(f"\nBOTTOM 10 OUTLETS BY REVENUE:")
print(q1_result.tail(10).to_string(index=False))
print(f"\nTotal outlets ranked: {len(q1_result)}")

📊 QUERY 1 — FRANCHISE REVENUE LEADERBOARD

TOP 10 OUTLETS BY REVENUE:
outlet_id      city archetype outlet_type    zone  total_revenue  avg_monthly_revenue  total_orders  avg_order_value  avg_net_margin_pct  avg_rating  revenue_rank
    OT080    Mumbai      star  Standalone   South     8394330.06            466351.67       14836.0           565.73               19.61        4.28             1
    OT016    Mumbai      star  Standalone    East     8345924.55            463662.48       14718.0           567.30               19.54        4.29             2
    OT084    Mumbai      star  Food Court Central     8262057.15            459003.18       14559.0           567.16               19.39        4.28             3
    OT015    Mumbai      star  Standalone   North     8239529.14            457751.62       14567.0           566.23               19.27        4.28             4
    OT033     Delhi      star  Standalone Central     7889052.51            438280.70       14285.0           551.8

In [3]:
# ============================================================
# QUERY 2 — ARCHETYPE PERFORMANCE BENCHMARKING
# Business Question: How do star vs stable vs struggling vs
# critical outlets compare across ALL key KPIs?
# ============================================================

q2 = """
SELECT
    archetype,
    COUNT(DISTINCT outlet_id)                              AS outlet_count,
    ROUND(AVG(revenue)::NUMERIC, 0)                        AS avg_monthly_revenue,
    ROUND(AVG(total_orders)::NUMERIC, 0)                   AS avg_monthly_orders,
    ROUND(AVG(avg_order_value)::NUMERIC, 0)                AS avg_order_value,
    ROUND(AVG(food_cost_pct)::NUMERIC, 1)                  AS avg_food_cost_pct,
    ROUND(AVG(contribution_margin_pct)::NUMERIC, 1)        AS avg_contribution_margin_pct,
    ROUND(AVG(net_margin_pct)::NUMERIC, 1)                 AS avg_net_margin_pct,
    ROUND(AVG(avg_rating)::NUMERIC, 2)                     AS avg_customer_rating,
    ROUND(AVG(complaint_rate)::NUMERIC, 1)                 AS avg_complaint_rate,
    ROUND(AVG(repeat_customer_pct)::NUMERIC, 1)            AS avg_repeat_customer_pct,
    ROUND(AVG(waste_rate)::NUMERIC, 1)                     AS avg_waste_rate,
    ROUND(AVG(upsell_rate)::NUMERIC, 1)                    AS avg_upsell_rate,
    ROUND(AVG(discount_pct)::NUMERIC, 1)                   AS avg_discount_pct
FROM monthly_performance
GROUP BY archetype
ORDER BY avg_monthly_revenue DESC
"""

q2_result = run_query(q2)

print("📊 QUERY 2 — ARCHETYPE PERFORMANCE BENCHMARKING")
print("="*65)
print(q2_result.to_string(index=False))

📊 QUERY 2 — ARCHETYPE PERFORMANCE BENCHMARKING
 archetype  outlet_count  avg_monthly_revenue  avg_monthly_orders  avg_order_value  avg_food_cost_pct  avg_contribution_margin_pct  avg_net_margin_pct  avg_customer_rating  avg_complaint_rate  avg_repeat_customer_pct  avg_waste_rate  avg_upsell_rate  avg_discount_pct
      star            25             345950.0               700.0            485.0               31.5                         45.7                18.2                 4.28                 7.1                     56.5            12.9             37.5               2.2
    stable            26             228980.0               468.0            477.0               31.5                         44.7                 3.7                 3.94                10.5                     50.1            18.5             33.0               3.0
       new             8             128328.0               306.0            412.0               31.5                         42.2               -12.

In [4]:
# ============================================================
# QUERY 3 — MONTHLY REVENUE TREND BY ARCHETYPE
# Business Question: Are struggling outlets getting worse
# over time or recovering? Show the trend.
# ============================================================

q3 = """
SELECT
    month,
    archetype,
    COUNT(DISTINCT outlet_id)                          AS outlets,
    ROUND(AVG(revenue)::NUMERIC, 0)                    AS avg_revenue,
    ROUND(AVG(net_margin_pct)::NUMERIC, 1)             AS avg_net_margin_pct,
    ROUND(AVG(avg_rating)::NUMERIC, 2)                 AS avg_rating,
    ROUND(AVG(complaint_rate)::NUMERIC, 1)             AS avg_complaint_rate,
    ROUND(AVG(repeat_customer_pct)::NUMERIC, 1)        AS avg_repeat_pct
FROM monthly_performance
GROUP BY month, archetype
ORDER BY month, archetype
"""

q3_result = run_query(q3)

print("📊 QUERY 3 — MONTHLY REVENUE TREND BY ARCHETYPE")
print("="*65)
print(f"Total rows: {len(q3_result)}")
print(f"\nSample — Critical outlets trend:")
print(q3_result[q3_result['archetype']=='critical'][
    ['month','avg_revenue','avg_net_margin_pct','avg_rating']
].to_string(index=False))

📊 QUERY 3 — MONTHLY REVENUE TREND BY ARCHETYPE
Total rows: 90

Sample — Critical outlets trend:
  month  avg_revenue  avg_net_margin_pct  avg_rating
2023-01      73402.0               -85.9        3.12
2023-02      61941.0              -108.5        3.09
2023-03      75359.0               -83.4        3.09
2023-04      75015.0               -84.9        3.08
2023-05      59842.0              -117.8        3.10
2023-06      62207.0              -109.4        3.10
2023-07      72486.0               -90.9        3.11
2023-08      72874.0               -89.6        3.11
2023-09      69262.0               -94.4        3.09
2023-10      85155.0               -69.6        3.09
2023-11      81320.0               -75.1        3.10
2023-12      86839.0               -66.9        3.09
2024-01      73227.0               -88.4        3.08
2024-02      68635.0               -97.5        3.10
2024-03      74275.0               -87.4        3.10
2024-04      70696.0               -91.7        3.11
202

In [5]:
# ============================================================
# QUERY 4 — AGGREGATOR DEPENDENCY ANALYSIS
# Business Question: How dependent are we on Zomato/Swiggy?
# What does it actually cost us in commission?
# ============================================================

q4 = """
SELECT
    om.archetype,
    dt.order_channel,
    COUNT(*)                                               AS total_orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) 
          OVER (PARTITION BY om.archetype), 1)             AS channel_share_pct,
    ROUND(AVG(dt.net_amount)::NUMERIC, 0)                  AS avg_order_value,
    ROUND(SUM(dt.aggregator_commission)::NUMERIC, 0)       AS total_commission_paid,
    ROUND(AVG(dt.aggregator_commission)::NUMERIC, 0)       AS avg_commission_per_order,
    ROUND(AVG(dt.contribution_margin)::NUMERIC, 0)         AS avg_contribution_margin,
    ROUND(AVG(dt.customer_rating)::NUMERIC, 2)             AS avg_rating
FROM daily_transactions dt
JOIN outlet_master om ON dt.outlet_id = om.outlet_id
GROUP BY om.archetype, dt.order_channel
ORDER BY om.archetype, total_orders DESC
"""

q4_result = run_query(q4)

print("📊 QUERY 4 — AGGREGATOR DEPENDENCY ANALYSIS")
print("="*65)
print(q4_result.to_string(index=False))

# Total commission paid to aggregators
total_commission = run_query("""
    SELECT 
        order_channel,
        ROUND(SUM(aggregator_commission)::NUMERIC, 0) AS total_commission,
        ROUND(SUM(net_amount)::NUMERIC, 0)            AS total_revenue,
        ROUND(SUM(aggregator_commission) * 100.0 / 
              SUM(net_amount), 1)                     AS commission_as_pct_revenue
    FROM daily_transactions
    WHERE order_channel IN ('Zomato','Swiggy')
    GROUP BY order_channel
    ORDER BY total_commission DESC
""")

print(f"\n💰 TOTAL AGGREGATOR COMMISSION PAID:")
print(total_commission.to_string(index=False))

📊 QUERY 4 — AGGREGATOR DEPENDENCY ANALYSIS
 archetype order_channel  total_orders  channel_share_pct  avg_order_value  total_commission_paid  avg_commission_per_order  avg_contribution_margin  avg_rating
  critical        Zomato         13911               55.2            461.0              1601983.0                     115.0                    170.0        3.10
  critical        Swiggy          9522               37.8            464.0              1016404.0                     107.0                    180.0        3.09
  critical       Dine-in          1005                4.0            468.0                    0.0                       0.0                    290.0        3.08
  critical    Direct App           495                2.0            444.0                11001.0                      22.0                    253.0        3.09
  critical      WhatsApp           261                1.0            484.0                 2527.0                      10.0                    290.0    

In [6]:
# ============================================================
# QUERY 5 — OUTLET RISK SCORING
# Business Question: Which outlets are at highest risk of
# failure in the next 60 days based on current signals?
# ============================================================

q5 = """
WITH outlet_kpis AS (
    SELECT
        mp.outlet_id,
        om.city,
        om.archetype,
        om.outlet_type,
        -- Revenue health
        ROUND(AVG(mp.revenue)::NUMERIC, 0)                  AS avg_revenue,
        ROUND(AVG(mp.net_margin_pct)::NUMERIC, 1)           AS avg_net_margin_pct,
        -- Last 3 months vs first 3 months revenue trend
        ROUND(AVG(CASE WHEN mp.month_num >= 16 
                       THEN mp.revenue END)::NUMERIC, 0)    AS last_3m_revenue,
        ROUND(AVG(CASE WHEN mp.month_num <= 3  
                       THEN mp.revenue END)::NUMERIC, 0)    AS first_3m_revenue,
        -- Operational signals
        ROUND(AVG(mp.complaint_rate)::NUMERIC, 1)           AS avg_complaint_rate,
        ROUND(AVG(mp.waste_rate)::NUMERIC, 1)               AS avg_waste_rate,
        ROUND(AVG(mp.avg_rating)::NUMERIC, 2)               AS avg_rating,
        ROUND(AVG(mp.repeat_customer_pct)::NUMERIC, 1)      AS avg_repeat_pct,
        -- Staff signals
        ROUND(AVG(sr.turnover_pct)::NUMERIC, 1)             AS avg_staff_turnover,
        SUM(sr.manager_change)                              AS total_manager_changes
    FROM monthly_performance mp
    JOIN outlet_master om  ON mp.outlet_id = om.outlet_id
    JOIN staff_records sr  ON mp.outlet_id = sr.outlet_id 
                           AND mp.month    = sr.month
    GROUP BY mp.outlet_id, om.city, om.archetype, om.outlet_type
),
risk_scored AS (
    SELECT *,
        -- Risk score: higher = more at risk (0-100)
        ROUND((
            -- Negative margin adds risk
            CASE WHEN avg_net_margin_pct < 0  THEN 25
                 WHEN avg_net_margin_pct < 5  THEN 15
                 WHEN avg_net_margin_pct < 10 THEN 8
                 ELSE 0 END
            +
            -- Revenue decline adds risk
            CASE WHEN last_3m_revenue < first_3m_revenue * 0.80 THEN 20
                 WHEN last_3m_revenue < first_3m_revenue * 0.90 THEN 12
                 ELSE 0 END
            +
            -- High complaints add risk
            CASE WHEN avg_complaint_rate > 20 THEN 20
                 WHEN avg_complaint_rate > 15 THEN 12
                 WHEN avg_complaint_rate > 10 THEN 6
                 ELSE 0 END
            +
            -- Low rating adds risk
            CASE WHEN avg_rating < 3.0 THEN 20
                 WHEN avg_rating < 3.5 THEN 12
                 WHEN avg_rating < 4.0 THEN 5
                 ELSE 0 END
            +
            -- High staff turnover adds risk
            CASE WHEN avg_staff_turnover > 15 THEN 15
                 WHEN avg_staff_turnover > 10 THEN 8
                 ELSE 0 END
        )::NUMERIC, 0) AS risk_score,
        CASE 
            WHEN avg_net_margin_pct < 0 
             AND avg_complaint_rate > 15 
             AND avg_rating < 3.5         THEN 'CRITICAL — Immediate Intervention'
            WHEN avg_net_margin_pct < 5  
             AND avg_complaint_rate > 12  THEN 'HIGH RISK — Action Within 2 Weeks'
            WHEN avg_net_margin_pct < 10 
             OR  avg_complaint_rate > 10  THEN 'MEDIUM RISK — Monitor Closely'
            ELSE                               'HEALTHY — Continue Support'
        END AS intervention_label
    FROM outlet_kpis
)
SELECT *
FROM risk_scored
ORDER BY risk_score DESC
"""

q5_result = run_query(q5)

print("📊 QUERY 5 — OUTLET RISK SCORING")
print("="*65)
print(f"\n🚨 TOP 15 HIGH RISK OUTLETS:")
print(q5_result.head(15)[['outlet_id','city','archetype',
                           'avg_net_margin_pct','avg_complaint_rate',
                           'avg_rating','avg_staff_turnover',
                           'risk_score','intervention_label']].to_string(index=False))

print(f"\n✅ HEALTHY OUTLETS: {len(q5_result[q5_result['intervention_label'].str.contains('HEALTHY')])}")
print(f"⚠️  MEDIUM RISK    : {len(q5_result[q5_result['intervention_label'].str.contains('MEDIUM')])}")
print(f"🔴 HIGH RISK       : {len(q5_result[q5_result['intervention_label'].str.contains('HIGH RISK')])}")
print(f"🚨 CRITICAL        : {len(q5_result[q5_result['intervention_label'].str.contains('CRITICAL')])}")

📊 QUERY 5 — OUTLET RISK SCORING

🚨 TOP 15 HIGH RISK OUTLETS:
outlet_id      city  archetype  avg_net_margin_pct  avg_complaint_rate  avg_rating  avg_staff_turnover  risk_score                intervention_label
    OT008   Kolkata   critical              -111.0                17.9        3.09                16.0        76.0 CRITICAL — Immediate Intervention
    OT081   Lucknow   critical               -78.0                18.4        3.09                15.4        76.0 CRITICAL — Immediate Intervention
    OT063      Pune   critical               -72.8                20.3        3.12                15.7        72.0 CRITICAL — Immediate Intervention
    OT075   Kolkata struggling               -40.1                15.6        3.49                11.4        69.0 CRITICAL — Immediate Intervention
    OT036    Jaipur   critical               -68.8                20.5        3.11                14.8        65.0 CRITICAL — Immediate Intervention
    OT074 Bangalore   critical              -

In [7]:
# ============================================================
# QUERY 6 — PEAK HOUR & DAY DEMAND ANALYSIS
# Business Question: When exactly do orders peak?
# Which hours and days drive the most revenue?
# ============================================================

q6 = """
SELECT
    EXTRACT(HOUR FROM transaction_time)         AS hour_of_day,
    day_of_week,
    COUNT(*)                                    AS total_orders,
    ROUND(AVG(net_amount)::NUMERIC, 0)          AS avg_order_value,
    ROUND(SUM(net_amount)::NUMERIC, 0)          AS total_revenue,
    ROUND(AVG(contribution_margin)::NUMERIC, 0) AS avg_contribution_margin,
    ROUND(AVG(customer_rating)::NUMERIC, 2)     AS avg_rating,
    SUM(had_complaint)                          AS total_complaints,
    ROUND(AVG(prep_time_mins)::NUMERIC, 1)      AS avg_prep_time
FROM daily_transactions
GROUP BY EXTRACT(HOUR FROM transaction_time), day_of_week
ORDER BY total_orders DESC
LIMIT 20
"""

q6_result = run_query(q6)

print("📊 QUERY 6 — PEAK HOUR & DAY DEMAND ANALYSIS")
print("="*65)
print("\nTOP 20 HIGHEST VOLUME HOUR-DAY COMBINATIONS:")
print(q6_result.to_string(index=False))

# Overall hourly summary
q6b = """
SELECT
    EXTRACT(HOUR FROM transaction_time)          AS hour_of_day,
    COUNT(*)                                     AS total_orders,
    ROUND(SUM(net_amount)::NUMERIC, 0)           AS total_revenue,
    ROUND(AVG(contribution_margin)::NUMERIC, 0)  AS avg_contribution_margin
FROM daily_transactions
GROUP BY EXTRACT(HOUR FROM transaction_time)
ORDER BY hour_of_day
"""
q6b_result = run_query(q6b)
print("\nHOURLY ORDER DISTRIBUTION (ALL DAYS):")
print(q6b_result.to_string(index=False))

📊 QUERY 6 — PEAK HOUR & DAY DEMAND ANALYSIS

TOP 20 HIGHEST VOLUME HOUR-DAY COMBINATIONS:
 hour_of_day day_of_week  total_orders  avg_order_value  total_revenue  avg_contribution_margin  avg_rating  total_complaints  avg_prep_time
        20.0      Sunday         10672            486.0      5187844.0                    216.0        4.00              1146           13.3
        20.0    Saturday         10355            481.0      4984867.0                    214.0        4.00              1002           13.3
        19.0      Sunday          9173            482.0      4421359.0                    214.0        3.99               903           13.2
        13.0      Sunday          9091            481.0      4375015.0                    214.0        3.99               928           13.2
        21.0      Sunday          9053            485.0      4389544.0                    216.0        4.00               890           13.3
        21.0    Saturday          8948            474.0      424

In [8]:
# ============================================================
# QUERY 7 — CITY TIER PERFORMANCE ANALYSIS
# Business Question: Are tier 2 and tier 3 cities worth
# expanding into? What is the ROI difference vs tier 1?
# ============================================================

q7 = """
SELECT
    om.tier,
    om.city,
    COUNT(DISTINCT om.outlet_id)                        AS outlet_count,
    ROUND(AVG(mp.revenue)::NUMERIC, 0)                  AS avg_monthly_revenue,
    ROUND(AVG(mp.total_orders)::NUMERIC, 0)             AS avg_monthly_orders,
    ROUND(AVG(mp.avg_order_value)::NUMERIC, 0)          AS avg_order_value,
    ROUND(AVG(mp.net_margin_pct)::NUMERIC, 1)           AS avg_net_margin_pct,
    ROUND(AVG(mp.rental_cost)::NUMERIC, 0)              AS avg_rental_cost,
    ROUND(AVG(mp.food_cost_pct)::NUMERIC, 1)            AS avg_food_cost_pct,
    ROUND(AVG(mp.avg_rating)::NUMERIC, 2)               AS avg_rating,
    ROUND(AVG(mp.repeat_customer_pct)::NUMERIC, 1)      AS avg_repeat_pct,
    ROUND(AVG(mp.complaint_rate)::NUMERIC, 1)           AS avg_complaint_rate,
    ROUND(AVG(mp.waste_rate)::NUMERIC, 1)               AS avg_waste_rate,
    -- Revenue per rupee of rent
    ROUND((AVG(mp.revenue) / AVG(mp.rental_cost))
          ::NUMERIC, 2)                                 AS revenue_per_rent_rupee
FROM monthly_performance mp
JOIN outlet_master om ON mp.outlet_id = om.outlet_id
GROUP BY om.tier, om.city
ORDER BY om.tier, avg_monthly_revenue DESC
"""

q7_result = run_query(q7)

print("📊 QUERY 7 — CITY TIER PERFORMANCE ANALYSIS")
print("="*65)
print(q7_result.to_string(index=False))

# Tier level summary
q7b = """
SELECT
    om.tier,
    COUNT(DISTINCT om.outlet_id)                        AS outlet_count,
    ROUND(AVG(mp.revenue)::NUMERIC, 0)                  AS avg_monthly_revenue,
    ROUND(AVG(mp.net_margin_pct)::NUMERIC, 1)           AS avg_net_margin_pct,
    ROUND(AVG(mp.rental_cost)::NUMERIC, 0)              AS avg_rental_cost,
    ROUND((AVG(mp.revenue) / 
           AVG(mp.rental_cost))::NUMERIC, 2)            AS revenue_per_rent_rupee,
    ROUND(AVG(mp.avg_rating)::NUMERIC, 2)               AS avg_rating
FROM monthly_performance mp
JOIN outlet_master om ON mp.outlet_id = om.outlet_id
GROUP BY om.tier
ORDER BY om.tier
"""
q7b_result = run_query(q7b)
print("\nTIER LEVEL SUMMARY:")
print(q7b_result.to_string(index=False))

📊 QUERY 7 — CITY TIER PERFORMANCE ANALYSIS
 tier       city  outlet_count  avg_monthly_revenue  avg_monthly_orders  avg_order_value  avg_net_margin_pct  avg_rental_cost  avg_food_cost_pct  avg_rating  avg_repeat_pct  avg_complaint_rate  avg_waste_rate  revenue_per_rent_rupee
    1     Mumbai            10             337909.0               599.0            563.0                -0.8         120000.0               31.5        3.95            50.3                10.5            18.2                    2.82
    1      Delhi             7             323509.0               595.0            541.0                -4.1         120000.0               31.5        3.96            50.8                10.2            18.0                    2.70
    1  Bangalore            16             258787.0               498.0            517.0               -13.2         120000.0               31.5        3.85            48.8                11.4            19.9                    2.16
    1  Hyderabad         

In [9]:
# ============================================================
# QUERY 8 — STAFF IMPACT ON PERFORMANCE
# Business Question: Does high staff turnover directly
# hurt revenue and ratings? Prove it with data.
# ============================================================

q8 = """
WITH staff_performance AS (
    SELECT
        sr.outlet_id,
        sr.month,
        sr.turnover_pct,
        sr.manager_change,
        sr.training_hours,
        sr.absenteeism_pct,
        mp.revenue,
        mp.avg_rating,
        mp.complaint_rate,
        mp.net_margin_pct,
        mp.repeat_customer_pct,
        -- Turnover buckets
        CASE
            WHEN sr.turnover_pct < 5  THEN '1. Low (<5%)'
            WHEN sr.turnover_pct < 10 THEN '2. Medium (5-10%)'
            WHEN sr.turnover_pct < 15 THEN '3. High (10-15%)'
            ELSE                           '4. Very High (>15%)'
        END AS turnover_bucket
    FROM staff_records sr
    JOIN monthly_performance mp 
      ON sr.outlet_id = mp.outlet_id 
     AND sr.month     = mp.month
)
SELECT
    turnover_bucket,
    COUNT(*)                                          AS months_observed,
    ROUND(AVG(revenue)::NUMERIC, 0)                   AS avg_revenue,
    ROUND(AVG(avg_rating)::NUMERIC, 2)                AS avg_rating,
    ROUND(AVG(complaint_rate)::NUMERIC, 1)            AS avg_complaint_rate,
    ROUND(AVG(net_margin_pct)::NUMERIC, 1)            AS avg_net_margin_pct,
    ROUND(AVG(repeat_customer_pct)::NUMERIC, 1)       AS avg_repeat_pct,
    ROUND(AVG(training_hours)::NUMERIC, 1)            AS avg_training_hours
FROM staff_performance
GROUP BY turnover_bucket
ORDER BY turnover_bucket
"""

q8_result = run_query(q8)

print("📊 QUERY 8 — STAFF TURNOVER IMPACT ON PERFORMANCE")
print("="*65)
print(q8_result.to_string(index=False))

# Manager change impact
q8b = """
SELECT
    sr.manager_change,
    COUNT(*)                                          AS observations,
    ROUND(AVG(mp.revenue)::NUMERIC, 0)                AS avg_revenue,
    ROUND(AVG(mp.avg_rating)::NUMERIC, 2)             AS avg_rating,
    ROUND(AVG(mp.complaint_rate)::NUMERIC, 1)         AS avg_complaint_rate,
    ROUND(AVG(mp.net_margin_pct)::NUMERIC, 1)         AS avg_net_margin_pct
FROM staff_records sr
JOIN monthly_performance mp 
  ON sr.outlet_id = mp.outlet_id 
 AND sr.month     = mp.month
GROUP BY sr.manager_change
ORDER BY sr.manager_change
"""
q8b_result = run_query(q8b)
print("\nMANAGER CHANGE IMPACT:")
print(q8b_result.to_string(index=False))

📊 QUERY 8 — STAFF TURNOVER IMPACT ON PERFORMANCE
    turnover_bucket  months_observed  avg_revenue  avg_rating  avg_complaint_rate  avg_net_margin_pct  avg_repeat_pct  avg_training_hours
       1. Low (<5%)              173     321349.0        4.21                 7.9                15.4            55.2                 9.0
  2. Medium (5-10%)              600     252958.0        4.00                 9.8                 3.7            51.3                 8.0
   3. High (10-15%)              591     175775.0        3.71                12.9               -19.4            46.3                 6.9
4. Very High (>15%)              166     105548.0        3.33                16.6               -60.2            39.3                 5.2

MANAGER CHANGE IMPACT:
 manager_change  observations  avg_revenue  avg_rating  avg_complaint_rate  avg_net_margin_pct
              0          1389     218881.0        3.85                11.4                -9.6
              1           141     175507.0     

In [12]:
# ============================================================
# QUERY 9 — COMPLAINT EARLY WARNING SYSTEM (FIXED)
# ============================================================

q9 = """
WITH monthly_complaints AS (
    SELECT
        outlet_id,
        month,
        month_num,
        complaint_rate,
        avg_rating,
        ROUND(AVG(complaint_rate) OVER (
            PARTITION BY outlet_id
            ORDER BY month_num
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        )::NUMERIC, 1) AS rolling_3m_complaint_rate,
        LAG(complaint_rate, 1) OVER (
            PARTITION BY outlet_id ORDER BY month_num
        ) AS prev_month_complaint_rate,
        LAG(avg_rating, 1) OVER (
            PARTITION BY outlet_id ORDER BY month_num
        ) AS prev_month_rating
    FROM monthly_performance
),
spike_detected AS (
    SELECT
        mc.*,
        om.city,
        om.archetype,
        ROUND((complaint_rate - prev_month_complaint_rate)
              ::NUMERIC, 1)                            AS complaint_mom_change,
        ROUND((avg_rating - prev_month_rating)
              ::NUMERIC, 2)                            AS rating_mom_change,
        CASE
            WHEN complaint_rate > rolling_3m_complaint_rate * 1.20
             AND prev_month_complaint_rate IS NOT NULL
            THEN 1 ELSE 0
        END AS complaint_spike_flag
    FROM monthly_complaints mc
    JOIN outlet_master om ON mc.outlet_id = om.outlet_id
)
SELECT
    outlet_id,
    city,
    archetype,
    month,
    ROUND(complaint_rate::NUMERIC, 1)               AS complaint_rate,
    ROUND(rolling_3m_complaint_rate::NUMERIC, 1)    AS rolling_3m_avg,
    ROUND(complaint_mom_change::NUMERIC, 1)         AS mom_change,
    ROUND(avg_rating::NUMERIC, 2)                   AS avg_rating,
    ROUND(rating_mom_change::NUMERIC, 2)            AS rating_change,
    complaint_spike_flag
FROM spike_detected
WHERE complaint_spike_flag = 1
ORDER BY complaint_mom_change DESC
LIMIT 20
"""

q9_result = run_query(q9)

print("📊 QUERY 9 — COMPLAINT EARLY WARNING SYSTEM (FIXED)")
print("="*65)
print(f"Total complaint spike events detected: {len(q9_result)}")
print(f"\nTOP 20 WORST COMPLAINT SPIKES:")
print(q9_result.to_string(index=False))

# Summary by archetype
q9b = """
WITH monthly_complaints AS (
    SELECT
        outlet_id,
        month_num,
        complaint_rate,
        ROUND(AVG(complaint_rate) OVER (
            PARTITION BY outlet_id
            ORDER BY month_num
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        )::NUMERIC, 1) AS rolling_3m_complaint_rate,
        LAG(complaint_rate,1) OVER (
            PARTITION BY outlet_id ORDER BY month_num
        ) AS prev_complaint
    FROM monthly_performance
)
SELECT
    om.archetype,
    COUNT(*) FILTER (
        WHERE mc.complaint_rate > mc.rolling_3m_complaint_rate * 1.20
        AND   mc.prev_complaint IS NOT NULL
    )                                               AS spike_events,
    COUNT(*)                                        AS total_months,
    ROUND(COUNT(*) FILTER (
        WHERE mc.complaint_rate > mc.rolling_3m_complaint_rate * 1.20
        AND   mc.prev_complaint IS NOT NULL
    ) * 100.0 / COUNT(*), 1)                        AS spike_frequency_pct
FROM monthly_complaints mc
JOIN outlet_master om ON mc.outlet_id = om.outlet_id
GROUP BY om.archetype
ORDER BY spike_frequency_pct DESC
"""
q9b_result = run_query(q9b)
print(f"\nSPIKE FREQUENCY BY ARCHETYPE:")
print(q9b_result.to_string(index=False))

📊 QUERY 9 — COMPLAINT EARLY WARNING SYSTEM (FIXED)
Total complaint spike events detected: 20

TOP 20 WORST COMPLAINT SPIKES:
outlet_id       city  archetype   month  complaint_rate  rolling_3m_avg  mom_change  avg_rating  rating_change  complaint_spike_flag
    OT081    Lucknow   critical 2023-02            25.9            20.8        10.3        3.08           0.00                     1
    OT008    Kolkata   critical 2023-04            18.4            14.3         8.7        3.10          -0.04                     1
    OT012       Pune        new 2023-04            17.1            12.8         8.5        3.67          -0.07                     1
    OT031    Lucknow struggling 2023-08            20.2            16.5         7.8        3.47          -0.12                     1
    OT004    Lucknow struggling 2024-06            17.2            13.8         7.6        3.51           0.12                     1
    OT074  Bangalore   critical 2023-05            19.4            14.6      

In [13]:
# ============================================================
# QUERY 10 — DISCOUNT ROI ANALYSIS
# Business Question: Are discounts actually driving more
# orders and revenue, or just burning margin?
# ============================================================

q10 = """
WITH discount_buckets AS (
    SELECT
        outlet_id,
        order_channel,
        net_amount,
        discount_applied,
        contribution_margin,
        customer_rating,
        is_repeat_customer,
        upsell_taken,
        CASE
            WHEN discount_applied = 0                        THEN '1. No Discount'
            WHEN discount_applied < net_amount * 0.10        THEN '2. Low (< 10%)'
            WHEN discount_applied < net_amount * 0.20        THEN '3. Medium (10-20%)'
            ELSE                                                  '4. High (> 20%)'
        END AS discount_bucket
    FROM daily_transactions
)
SELECT
    discount_bucket,
    COUNT(*)                                              AS total_orders,
    ROUND(AVG(net_amount)::NUMERIC, 0)                    AS avg_net_order_value,
    ROUND(AVG(discount_applied)::NUMERIC, 0)              AS avg_discount_amount,
    ROUND(AVG(contribution_margin)::NUMERIC, 0)           AS avg_contribution_margin,
    ROUND(AVG(customer_rating)::NUMERIC, 2)               AS avg_rating,
    ROUND(AVG(is_repeat_customer) * 100::NUMERIC, 1)      AS repeat_customer_pct,
    ROUND(AVG(upsell_taken) * 100::NUMERIC, 1)            AS upsell_rate_pct,
    -- Contribution margin as % of net amount
    ROUND((AVG(contribution_margin) / 
           NULLIF(AVG(net_amount), 0) * 100)::NUMERIC, 1) AS contribution_margin_pct
FROM discount_buckets
GROUP BY discount_bucket
ORDER BY discount_bucket
"""

q10_result = run_query(q10)

print("📊 QUERY 10 — DISCOUNT ROI ANALYSIS")
print("="*65)
print(q10_result.to_string(index=False))

📊 QUERY 10 — DISCOUNT ROI ANALYSIS
   discount_bucket  total_orders  avg_net_order_value  avg_discount_amount  avg_contribution_margin  avg_rating  repeat_customer_pct  upsell_rate_pct  contribution_margin_pct
    1. No Discount        538555                499.0                  0.0                    223.0        4.03                 51.9             34.3                     44.8
    2. Low (< 10%)         29333                461.0                 35.0                    200.0        3.88                 49.4             32.7                     43.5
3. Medium (10-20%)         54303                431.0                 64.0                    185.0        3.88                 49.5             32.4                     42.9
   4. High (> 20%)         59758                389.0                102.0                    165.0        3.88                 49.3             32.3                     42.3


In [14]:
# ============================================================
# QUERY 11 — ITEM CATEGORY PERFORMANCE ANALYSIS
# Business Question: Which food categories drive the most
# margin? Should we push combos or individual items?
# ============================================================

q11 = """
SELECT
    dt.item_category,
    om.outlet_type,
    COUNT(*)                                              AS total_orders,
    ROUND(AVG(dt.net_amount)::NUMERIC, 0)                 AS avg_order_value,
    ROUND(AVG(dt.food_cost)::NUMERIC, 0)                  AS avg_food_cost,
    ROUND(AVG(dt.contribution_margin)::NUMERIC, 0)        AS avg_contribution_margin,
    ROUND((AVG(dt.contribution_margin) /
           NULLIF(AVG(dt.net_amount),0) * 100)
          ::NUMERIC, 1)                                   AS contribution_margin_pct,
    ROUND(AVG(dt.customer_rating)::NUMERIC, 2)            AS avg_rating,
    ROUND(AVG(dt.upsell_taken) * 100::NUMERIC, 1)         AS upsell_rate_pct,
    ROUND(SUM(dt.contribution_margin)::NUMERIC, 0)        AS total_contribution
FROM daily_transactions dt
JOIN outlet_master om ON dt.outlet_id = om.outlet_id
GROUP BY dt.item_category, om.outlet_type
ORDER BY avg_contribution_margin DESC
"""

q11_result = run_query(q11)

print("📊 QUERY 11 — ITEM CATEGORY PERFORMANCE ANALYSIS")
print("="*65)
print(q11_result.to_string(index=False))

# Overall category summary
q11b = """
SELECT
    item_category,
    COUNT(*)                                              AS total_orders,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER()
          ::NUMERIC, 1)                                   AS order_share_pct,
    ROUND(AVG(net_amount)::NUMERIC, 0)                    AS avg_order_value,
    ROUND(AVG(contribution_margin)::NUMERIC, 0)           AS avg_contribution_margin,
    ROUND((AVG(contribution_margin) /
           NULLIF(AVG(net_amount),0) * 100)
          ::NUMERIC, 1)                                   AS margin_pct,
    ROUND(SUM(contribution_margin)::NUMERIC, 0)           AS total_contribution,
    ROUND(SUM(contribution_margin) * 100.0 /
          SUM(SUM(contribution_margin)) OVER()
          ::NUMERIC, 1)                                   AS contribution_share_pct
FROM daily_transactions
GROUP BY item_category
ORDER BY total_contribution DESC
"""

q11b_result = run_query(q11b)
print("\nOVERALL CATEGORY CONTRIBUTION SUMMARY:")
print(q11b_result.to_string(index=False))

📊 QUERY 11 — ITEM CATEGORY PERFORMANCE ANALYSIS
item_category outlet_type  total_orders  avg_order_value  avg_food_cost  avg_contribution_margin  contribution_margin_pct  avg_rating  upsell_rate_pct  total_contribution
        Combo     Highway         20466            838.0          276.0                    371.0                     44.3        4.09             35.3           7597083.0
        Combo        Mall         18342            830.0          274.0                    364.0                     43.9        3.97             33.1           6680416.0
        Combo  Standalone         31282            812.0          268.0                    353.0                     43.5        3.93             32.7          11047714.0
        Combo  Food Court         11391            790.0          261.0                    349.0                     44.2        4.06             34.9           3980732.0
        Pizza     Highway         30473            699.0          210.0                    329.0 

In [15]:
# ============================================================
# QUERY 12 — WEEKEND VS WEEKDAY REVENUE INTELLIGENCE
# Business Question: How much more do we earn on weekends?
# Should we staff up or run special menus on weekends?
# ============================================================

q12 = """
SELECT
    om.archetype,
    om.city,
    CASE WHEN dt.is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
    COUNT(*)                                               AS total_orders,
    ROUND(AVG(dt.net_amount)::NUMERIC, 0)                  AS avg_order_value,
    ROUND(SUM(dt.net_amount)::NUMERIC, 0)                  AS total_revenue,
    ROUND(AVG(dt.contribution_margin)::NUMERIC, 0)         AS avg_contribution_margin,
    ROUND(AVG(dt.customer_rating)::NUMERIC, 2)             AS avg_rating,
    ROUND(AVG(dt.is_repeat_customer) * 100::NUMERIC, 1)    AS repeat_customer_pct,
    ROUND(AVG(dt.upsell_taken) * 100::NUMERIC, 1)          AS upsell_rate_pct,
    SUM(dt.had_complaint)                                  AS total_complaints
FROM daily_transactions dt
JOIN outlet_master om ON dt.outlet_id = om.outlet_id
GROUP BY om.archetype, om.city, day_type
ORDER BY om.archetype, om.city, day_type
"""

q12_result = run_query(q12)

# Weekend vs Weekday summary
q12b = """
SELECT
    CASE WHEN is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END  AS day_type,
    COUNT(*)                                                     AS total_orders,
    ROUND(AVG(net_amount)::NUMERIC, 0)                           AS avg_order_value,
    ROUND(SUM(net_amount)::NUMERIC, 0)                           AS total_revenue,
    ROUND(AVG(contribution_margin)::NUMERIC, 0)                  AS avg_contribution_margin,
    ROUND(AVG(customer_rating)::NUMERIC, 2)                      AS avg_rating,
    ROUND(AVG(is_repeat_customer) * 100::NUMERIC, 1)             AS repeat_pct,
    ROUND(AVG(upsell_taken) * 100::NUMERIC, 1)                   AS upsell_pct,
    ROUND(AVG(prep_time_mins)::NUMERIC, 1)                       AS avg_prep_time,
    ROUND(AVG(delivery_time_mins)::NUMERIC, 1)                   AS avg_delivery_time
FROM daily_transactions
GROUP BY day_type
ORDER BY day_type
"""

q12b_result = run_query(q12b)

print("📊 QUERY 12 — WEEKEND VS WEEKDAY INTELLIGENCE")
print("="*65)
print("OVERALL WEEKEND VS WEEKDAY SUMMARY:")
print(q12b_result.to_string(index=False))
print(f"\nDetailed breakdown rows: {len(q12_result)}")
print(f"\nSample — Star outlets weekend vs weekday:")
print(q12_result[q12_result['archetype']=='star'][
    ['city','day_type','total_orders','avg_order_value',
     'avg_contribution_margin','avg_rating']
].head(16).to_string(index=False))

📊 QUERY 12 — WEEKEND VS WEEKDAY INTELLIGENCE
OVERALL WEEKEND VS WEEKDAY SUMMARY:
day_type  total_orders  avg_order_value  total_revenue  avg_contribution_margin  avg_rating  repeat_pct  upsell_pct  avg_prep_time  avg_delivery_time
 Weekday        456911            482.0    220199341.0                    214.0         4.0        51.4        33.9           13.2               27.4
 Weekend        225038            482.0    108572462.0                    214.0         4.0        51.4        33.9           13.3               27.4

Detailed breakdown rows: 88

Sample — Star outlets weekend vs weekday:
     city day_type  total_orders  avg_order_value  avg_contribution_margin  avg_rating
Ahmedabad  Weekday         22632            442.0                    201.0        4.28
Ahmedabad  Weekend         11103            444.0                    203.0        4.29
Bangalore  Weekday         37034            526.0                    240.0        4.29
Bangalore  Weekend         18328            525.0

In [16]:
# ============================================================
# QUERY 13 — CUSTOMER LOYALTY & RETENTION ANALYSIS
# Business Question: Which outlets are best at retaining
# customers? What drives repeat purchase behaviour?
# ============================================================

q13 = """
SELECT
    om.outlet_id,
    om.city,
    om.archetype,
    om.outlet_type,
    COUNT(*)                                               AS total_orders,
    SUM(dt.is_repeat_customer)                             AS repeat_orders,
    SUM(dt.is_new_customer)                                AS new_orders,
    ROUND(AVG(dt.is_repeat_customer) * 100::NUMERIC, 1)    AS repeat_customer_pct,
    ROUND(AVG(dt.net_amount)::NUMERIC, 0)                  AS avg_order_value,
    ROUND(AVG(dt.customer_rating)::NUMERIC, 2)             AS avg_rating,
    ROUND(AVG(dt.upsell_taken) * 100::NUMERIC, 1)          AS upsell_rate,
    ROUND(AVG(dt.had_complaint) * 100::NUMERIC, 1)         AS complaint_rate,
    -- Repeat customer revenue share
    ROUND(SUM(CASE WHEN dt.is_repeat_customer = 1
                   THEN dt.net_amount ELSE 0 END) * 100.0 /
          NULLIF(SUM(dt.net_amount), 0)::NUMERIC, 1)       AS repeat_revenue_share_pct,
    RANK() OVER (ORDER BY AVG(dt.is_repeat_customer) DESC) AS loyalty_rank
FROM daily_transactions dt
JOIN outlet_master om ON dt.outlet_id = om.outlet_id
GROUP BY om.outlet_id, om.city, om.archetype, om.outlet_type
ORDER BY repeat_customer_pct DESC
"""

q13_result = run_query(q13)

print("📊 QUERY 13 — CUSTOMER LOYALTY & RETENTION ANALYSIS")
print("="*65)
print("TOP 15 OUTLETS BY CUSTOMER LOYALTY:")
print(q13_result.head(15)[['outlet_id','city','archetype',
                             'repeat_customer_pct','avg_rating',
                             'upsell_rate','complaint_rate',
                             'repeat_revenue_share_pct',
                             'loyalty_rank']].to_string(index=False))

print("\nBOTTOM 15 OUTLETS BY CUSTOMER LOYALTY:")
print(q13_result.tail(15)[['outlet_id','city','archetype',
                             'repeat_customer_pct','avg_rating',
                             'upsell_rate','complaint_rate',
                             'repeat_revenue_share_pct',
                             'loyalty_rank']].to_string(index=False))

# Loyalty by archetype
q13b = """
SELECT
    om.archetype,
    ROUND(AVG(dt.is_repeat_customer) * 100::NUMERIC, 1)   AS avg_repeat_pct,
    ROUND(AVG(dt.net_amount)::NUMERIC, 0)                  AS avg_order_value,
    ROUND(SUM(CASE WHEN dt.is_repeat_customer = 1
                   THEN dt.net_amount ELSE 0 END) * 100.0 /
          NULLIF(SUM(dt.net_amount), 0)::NUMERIC, 1)       AS repeat_revenue_share
FROM daily_transactions dt
JOIN outlet_master om ON dt.outlet_id = om.outlet_id
GROUP BY om.archetype
ORDER BY avg_repeat_pct DESC
"""
q13b_result = run_query(q13b)
print("\nLOYALTY BY ARCHETYPE:")
print(q13b_result.to_string(index=False))

📊 QUERY 13 — CUSTOMER LOYALTY & RETENTION ANALYSIS
TOP 15 OUTLETS BY CUSTOMER LOYALTY:
outlet_id      city archetype  repeat_customer_pct  avg_rating  upsell_rate  complaint_rate  repeat_revenue_share_pct  loyalty_rank
    OT007   Kolkata      star                 57.6        4.28         37.6             7.4                      58.0             1
    OT050    Indore      star                 57.3        4.28         36.9             7.1                      57.3             2
    OT069      Pune      star                 57.1        4.28         37.9             7.1                      57.0             3
    OT058      Pune      star                 57.1        4.28         37.3             7.1                      57.0             4
    OT022 Hyderabad      star                 56.9        4.29         37.5             7.1                      56.6             5
    OT041 Hyderabad      star                 56.8        4.28         38.3             7.5                      57.1    

In [17]:
# ============================================================
# QUERY 14 — OUTLET TYPE ROI COMPARISON
# Business Question: Which outlet format — Mall, Standalone,
# Food Court, or Highway — gives the best return?
# ============================================================

q14 = """
SELECT
    om.outlet_type,
    om.tier,
    COUNT(DISTINCT om.outlet_id)                           AS outlet_count,
    ROUND(AVG(mp.revenue)::NUMERIC, 0)                     AS avg_monthly_revenue,
    ROUND(AVG(mp.total_orders)::NUMERIC, 0)                AS avg_monthly_orders,
    ROUND(AVG(mp.avg_order_value)::NUMERIC, 0)             AS avg_order_value,
    ROUND(AVG(mp.net_margin_pct)::NUMERIC, 1)              AS avg_net_margin_pct,
    ROUND(AVG(mp.contribution_margin_pct)::NUMERIC, 1)     AS avg_contribution_pct,
    ROUND(AVG(mp.rental_cost)::NUMERIC, 0)                 AS avg_rental_cost,
    ROUND(AVG(mp.avg_rating)::NUMERIC, 2)                  AS avg_rating,
    ROUND(AVG(mp.repeat_customer_pct)::NUMERIC, 1)         AS avg_repeat_pct,
    ROUND(AVG(mp.waste_rate)::NUMERIC, 1)                  AS avg_waste_rate,
    ROUND(AVG(mp.upsell_rate)::NUMERIC, 1)                 AS avg_upsell_rate,
    -- Revenue efficiency
    ROUND((AVG(mp.revenue) /
           NULLIF(AVG(mp.rental_cost), 0))::NUMERIC, 2)    AS revenue_per_rent_rupee,
    -- Annual net margin projection
    ROUND(AVG(mp.net_margin) * 12::NUMERIC, 0)             AS projected_annual_net_margin
FROM monthly_performance mp
JOIN outlet_master om ON mp.outlet_id = om.outlet_id
GROUP BY om.outlet_type, om.tier
ORDER BY avg_net_margin_pct DESC
"""

q14_result = run_query(q14)

print("📊 QUERY 14 — OUTLET TYPE ROI COMPARISON")
print("="*65)
print(q14_result.to_string(index=False))

# Overall outlet type summary
q14b = """
SELECT
    om.outlet_type,
    COUNT(DISTINCT om.outlet_id)                           AS outlet_count,
    ROUND(AVG(mp.revenue)::NUMERIC, 0)                     AS avg_monthly_revenue,
    ROUND(AVG(mp.net_margin_pct)::NUMERIC, 1)              AS avg_net_margin_pct,
    ROUND((AVG(mp.revenue) /
           NULLIF(AVG(mp.rental_cost),0))::NUMERIC, 2)     AS revenue_per_rent_rupee,
    ROUND(AVG(mp.avg_rating)::NUMERIC, 2)                  AS avg_rating
FROM monthly_performance mp
JOIN outlet_master om ON mp.outlet_id = om.outlet_id
GROUP BY om.outlet_type
ORDER BY avg_net_margin_pct DESC
"""
q14b_result = run_query(q14b)
print("\nOVERALL OUTLET TYPE SUMMARY:")
print(q14b_result.to_string(index=False))

📊 QUERY 14 — OUTLET TYPE ROI COMPARISON
outlet_type  tier  outlet_count  avg_monthly_revenue  avg_monthly_orders  avg_order_value  avg_net_margin_pct  avg_contribution_pct  avg_rental_cost  avg_rating  avg_repeat_pct  avg_waste_rate  avg_upsell_rate  revenue_per_rent_rupee  projected_annual_net_margin
    Highway     2             6             234354.0               536.0            435.0                 9.1                  44.9          75000.0        4.13            53.7            14.9             35.6                    3.12                     370041.0
 Food Court     3             3             121250.0               355.0            341.0                 3.6                  44.2          45000.0        3.98            50.5            17.7             33.1                    2.69                     108079.0
    Highway     1            10             299292.0               560.0            532.0                -1.2                  44.2         120000.0        3.96           

In [20]:
# ============================================================
# QUERY 15 — FRANCHISE INTERVENTION PRIORITY MATRIX
# Business Question: Final consolidated view — every outlet
# ranked, scored, and assigned an intervention action.
# This is the CORE deliverable of the entire project.
# ============================================================

q15 = """
WITH outlet_kpis AS (
    SELECT
        mp.outlet_id,
        om.city,
        om.tier,
        om.archetype,
        om.outlet_type,
        om.zone,
        om.outlet_age_months,
        -- Revenue metrics
        ROUND(SUM(mp.revenue)::NUMERIC, 0)                  AS total_revenue_18m,
        ROUND(AVG(mp.revenue)::NUMERIC, 0)                  AS avg_monthly_revenue,
        ROUND(AVG(mp.total_orders)::NUMERIC, 0)             AS avg_monthly_orders,
        ROUND(AVG(mp.avg_order_value)::NUMERIC, 0)          AS avg_order_value,
        -- Margin metrics
        ROUND(AVG(mp.net_margin_pct)::NUMERIC, 1)           AS avg_net_margin_pct,
        ROUND(AVG(mp.contribution_margin_pct)::NUMERIC, 1)  AS avg_contribution_pct,
        ROUND(AVG(mp.food_cost_pct)::NUMERIC, 1)            AS avg_food_cost_pct,
        -- Customer metrics
        ROUND(AVG(mp.avg_rating)::NUMERIC, 2)               AS avg_rating,
        ROUND(AVG(mp.complaint_rate)::NUMERIC, 1)           AS avg_complaint_rate,
        ROUND(AVG(mp.repeat_customer_pct)::NUMERIC, 1)      AS avg_repeat_pct,
        ROUND(AVG(mp.waste_rate)::NUMERIC, 1)               AS avg_waste_rate,
        ROUND(AVG(mp.upsell_rate)::NUMERIC, 1)              AS avg_upsell_rate,
        -- Staff metrics
        ROUND(AVG(sr.turnover_pct)::NUMERIC, 1)             AS avg_staff_turnover,
        SUM(sr.manager_change)                              AS total_manager_changes,
        ROUND(AVG(sr.training_hours)::NUMERIC, 1)           AS avg_training_hours,
        -- Trend: last 3 months revenue vs first 3 months
        ROUND(AVG(CASE WHEN mp.month_num >= 16
                       THEN mp.revenue END)::NUMERIC, 0)    AS last_3m_avg_revenue,
        ROUND(AVG(CASE WHEN mp.month_num <= 3
                       THEN mp.revenue END)::NUMERIC, 0)    AS first_3m_avg_revenue
    FROM monthly_performance mp
    JOIN outlet_master om ON mp.outlet_id = om.outlet_id
    JOIN staff_records sr ON mp.outlet_id = sr.outlet_id
                          AND mp.month    = sr.month
    GROUP BY mp.outlet_id, om.city, om.tier, om.archetype,
             om.outlet_type, om.zone, om.outlet_age_months
),
scored AS (
    SELECT *,
        -- Revenue trend direction
        ROUND(((last_3m_avg_revenue - first_3m_avg_revenue) /
               NULLIF(first_3m_avg_revenue, 0)) * 100
              ::NUMERIC, 1)                                  AS revenue_trend_pct,
        -- Composite performance score (0-100)
        ROUND((
            -- Revenue score (25 pts)
            LEAST(25, (avg_monthly_revenue / 300000.0) * 25)
            +
            -- Margin score (25 pts)
            CASE WHEN avg_net_margin_pct > 15  THEN 25
                 WHEN avg_net_margin_pct > 5   THEN 18
                 WHEN avg_net_margin_pct > 0   THEN 10
                 WHEN avg_net_margin_pct > -20 THEN 5
                 ELSE 0 END
            +
            -- Customer score (25 pts)
            ((avg_rating / 5.0) * 12.5) +
            (LEAST(1, avg_repeat_pct / 60.0) * 7.5) +
            (GREATEST(0, 1 - avg_complaint_rate / 25.0) * 5)
            +
            -- Operations score (25 pts)
            (GREATEST(0, 1 - avg_waste_rate / 35.0) * 10) +
            (GREATEST(0, 1 - avg_staff_turnover / 20.0) * 10) +
            (LEAST(1, avg_training_hours / 10.0) * 5)
        )::NUMERIC, 1)                                       AS composite_score
    FROM outlet_kpis
),
final_scored AS (
    SELECT *,
        RANK() OVER (ORDER BY composite_score DESC)          AS performance_rank,
        CASE
            WHEN composite_score >= 65  THEN 'TIER 1 — Star Performer'
            WHEN composite_score >= 45  THEN 'TIER 2 — Stable Operator'
            WHEN composite_score >= 30  THEN 'TIER 3 — Needs Improvement'
            ELSE                             'TIER 4 — Critical Risk'
        END                                                  AS performance_tier,
        CASE
            WHEN composite_score < 30
             AND avg_net_margin_pct < -20  THEN 'IMMEDIATE — Exit or Restructure'
            WHEN composite_score < 30      THEN 'URGENT — Intervene Within 2 Weeks'
            WHEN composite_score < 45      THEN 'HIGH — Action Plan Within 1 Month'
            WHEN composite_score < 65      THEN 'MEDIUM — Monthly Review'
            ELSE                                'LOW — Replicate Best Practices'
        END                                                  AS intervention_action
    FROM scored
)
SELECT *
FROM final_scored
ORDER BY composite_score DESC
"""

q15_result = run_query(q15)

print("📊 QUERY 15 — FRANCHISE INTERVENTION PRIORITY MATRIX")
print("="*65)
print(f"\nTOP 10 PERFORMING OUTLETS:")
print(q15_result.head(10)[['outlet_id','city','archetype',
                             'composite_score','avg_net_margin_pct',
                             'avg_rating','performance_tier',
                             'intervention_action']].to_string(index=False))

print(f"\nBOTTOM 10 OUTLETS — NEED IMMEDIATE ATTENTION:")
print(q15_result.tail(10)[['outlet_id','city','archetype',
                             'composite_score','avg_net_margin_pct',
                             'avg_rating','performance_tier',
                             'intervention_action']].to_string(index=False))

print(f"\nINTERVENTION SUMMARY:")
print(q15_result['intervention_action'].value_counts().to_string())

print(f"\nPERFORMANCE TIER DISTRIBUTION:")
print(q15_result['performance_tier'].value_counts().to_string())

# Save this as the master output file
q15_result.to_csv(
    r'D:\Projects\End-to-end projects\12. Franchise Intelligence System\Data\Processed\outlet_scores.csv',
    index=False
)
print(f"\n✅ outlet_scores.csv saved to Data/Processed folder")
print(f"Total outlets scored: {len(q15_result)}")

📊 QUERY 15 — FRANCHISE INTERVENTION PRIORITY MATRIX

TOP 10 PERFORMING OUTLETS:
outlet_id      city archetype  composite_score  avg_net_margin_pct  avg_rating        performance_tier            intervention_action
    OT033     Delhi      star             89.4                18.0        4.28 TIER 1 — Star Performer LOW — Replicate Best Practices
    OT011      Pune      star             89.3                21.6        4.27 TIER 1 — Star Performer LOW — Replicate Best Practices
    OT080    Mumbai      star             89.2                19.6        4.28 TIER 1 — Star Performer LOW — Replicate Best Practices
    OT027     Delhi      star             89.2                17.8        4.29 TIER 1 — Star Performer LOW — Replicate Best Practices
    OT015    Mumbai      star             89.0                19.3        4.28 TIER 1 — Star Performer LOW — Replicate Best Practices
    OT058      Pune      star             88.9                21.6        4.28 TIER 1 — Star Performer LOW — Replica